In [1]:
import pandas as pd
import tensorflow as tf
from transformers import BertTokenizer, TFBertForSequenceClassification
from sklearn.preprocessing import LabelEncoder

In [2]:
# 读取训练集和测试集
train_df = pd.read_csv("/content/Train-d.csv")
test_df = pd.read_csv("/content/Test-d.csv")

# 进行标签编码（将难度等级转换为数值）
label_encoder = LabelEncoder()
train_df["label"] = label_encoder.fit_transform(train_df["ques_difficulty"])
test_df["label"] = label_encoder.transform(test_df["ques_difficulty"])

# 获取文本和标签
train_texts, train_labels = train_df["ques_content"].tolist(), train_df["label"].tolist()
test_texts, test_labels = test_df["ques_content"].tolist(), test_df["label"].tolist()

In [3]:
MODEL_NAME = "bert-base-chinese"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# Tokenization
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

# 转换为 TensorFlow Dataset
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings), train_labels)).shuffle(1000).batch(16)
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encodings), test_labels)).batch(16)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/269k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/624 [00:00<?, ?B/s]

In [4]:
num_labels = len(label_encoder.classes_)

model = TFBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# 训练模型
model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=10
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

All PyTorch model weights were used when initializing TFBertForSequenceClassification.

Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/10
113/113 [==============================] - 98s 450ms/step - loss: 1.1303 - accuracy: 0.5025 - val_loss: 1.0372 - val_accuracy: 0.5863
Epoch 2/10
113/113 [==============================] - 49s 429ms/step - loss: 1.0298 - accuracy: 0.5618 - val_loss: 0.9874 - val_accuracy: 0.5597
Epoch 3/10
113/113 [==============================] - 48s 428ms/step - loss: 0.9949 - accuracy: 0.5679 - val_loss: 0.9788 - val_accuracy: 0.5664
Epoch 4/10
113/113 [==============================] - 48s 429ms/step - loss: 0.9068 - accuracy: 0.6127 - val_loss: 1.0244 - val_accuracy: 0.5796
Epoch 5/10
113/113 [==============================] - 49s 432ms/step - loss: 0.8371 - accuracy: 0.6587 - val_loss: 1.0175 - val_accuracy: 0.5796
Epoch 6/10
113/113 [==============================] - 49s 431ms/step - loss: 0.6290 - accuracy: 0.7618 - val_loss: 1.2945 - val_accuracy: 0.4845
Epoch 7/10
113/113 [==============================] - 48s 428ms/step - loss: 0.4239 - accuracy: 0.8499 - val_loss: 1.4158 - val_ac

In [5]:
# 评估模型
loss, accuracy = model.evaluate(test_dataset)
print(f"Test Accuracy: {accuracy:.4f}")

29/29 [==============================] - 4s 137ms/step - loss: 1.8582 - accuracy: 0.4867
Test Accuracy: 0.4867


In [8]:
# 保存模型
model.save_pretrained("bert_difficulty_classifier")
tokenizer.save_pretrained("bert_difficulty_classifier")

# 单个样本预测函数
def predict(text):
    inputs = tokenizer(text, return_tensors="tf", truncation=True, padding=True, max_length=128)
    logits = model(**inputs).logits
    predicted_label = tf.argmax(logits, axis=1).numpy()[0]
    return label_encoder.inverse_transform([predicted_label])[0]

# 示例预测
print(predict("如果200页纸的总厚度为150厘米那么每张纸的厚度是多少毫米"))

容易
